# 교과서 67~81쪽 붓꽃 k-최근접 이웃 분류 (수업용 사본)

- 교과서 소스 코드 노트북을 그대로 옮기고, 데이터 파일을 인터넷 주소에서 바로 읽도록 바꿨어요.
- 혼동행렬 셀은 교과서 80쪽 코드처럼 `labels=classNames`를 넣어 행·열 이름표가 맞게 했어요.
- 위에서부터 차례로 ▶(Shift+Enter)를 눌러 실행해요.

# 인공지능 기초 67~72쪽 데이터 전처리

파일 업로드하기

교과서는 이 자리에서 `files.upload()`로 Iris1.csv, Iris2.csv를 올립니다.
이 사본은 수업 저장소에 올려 둔 파일을 아래 셀에서 바로 읽으므로 **올리는 과정 없이** 다음 셀부터 실행하면 됩니다.

데이터셋 불러오기

In [ ]:
import pandas as pd # pandas 모듈을 불러와 별칭을 pd로 지정
df1 = pd.read_csv('https://raw.githubusercontent.com/sweet1203/ds_ai_class/main/lessons/u2-train-eval/assets/iris1.csv', encoding = 'cp949') # 데이터프레임으로 변환
df2 = pd.read_csv('https://raw.githubusercontent.com/sweet1203/ds_ai_class/main/lessons/u2-train-eval/assets/iris2.csv', encoding = 'cp949')
df1.head() # df1 앞 5개 데이터 출력

두 개의 데이터 파일 합치기

In [ ]:
df = pd.concat([df1, df2]) # df1과 df2를 위아래 방향으로 합치기
df

결측치 확인하기

In [ ]:
df.isna().sum()

결측치 처리하기

In [ ]:
df3 = df.dropna(axis = 0, inplace = False)
df3

기초 통계량으로 이상치 확인하기

In [ ]:
df3.describe()

In [ ]:
df3.groupby('종류')['꽃받침 길이'].describe()

히스토그램으로 이상치 확인하기

In [ ]:
!pip install koreanize-matplotlib # 한글 폰트 설정 라이브러리 설치

In [ ]:
import koreanize_matplotlib # 한글 출력 가능한 그래프 모듈
import matplotlib.pyplot as plt # 그래프 그리기에 필요한 모듈
import seaborn as sns # 시각화하기 위해 seaborn 모듈 불러오기

In [ ]:
sns.histplot(data = df3, x = '꽃받침 길이', hue = '종류')

In [ ]:
sns.histplot(data = df3, x = '꽃받침 너비', hue = '종류')

In [ ]:
sns.histplot(data = df3, x = '꽃잎 길이', hue = '종류')

In [ ]:
sns.histplot(data = df3, x = '꽃잎 너비', hue = '종류')

박스플롯으로 이상치 확인하기

In [ ]:
sns.catplot(y = '꽃받침 길이', x = '종류', kind = 'box', data = df3, hue ='종류')
sns.catplot(y = '꽃받침 너비', x = '종류', kind = 'box', data = df3, hue = '종류')
sns.catplot(y = '꽃잎 길이', x = '종류', kind = 'box', data = df3, hue = '종류')
sns.catplot(y = '꽃잎 너비', x = '종류', kind = 'box', data = df3, hue = '종류')

이상치 제거하기

(1) 사분위수 범위

In [ ]:
Q1 = df3.groupby('종류')['꽃받침 길이'].quantile(0.25) # 제1사분위수
Q3 = df3.groupby('종류')['꽃받침 길이'].quantile(0.75) # 제3사분위수
IQR = Q3 - Q1 # 사분위수 범위
print('Q1:', Q1); print('Q3:', Q3); print('IQR:', IQR)

(2) 안팎 울타리값

In [ ]:
lower = Q1 - 1.5 * IQR; upper = Q3 + 1.5 * IQR
print('lower:', lower); print('upper:', upper)

(3) 이상치 찾기

In [ ]:
df4 = df3[df3['종류'] == '버시컬러']
df4[(df4['꽃받침 길이'] < lower['버시컬러'] ) | (df4['꽃받침 길이'] > upper['버시컬러'])]

In [ ]:
#이상치 찾기 함수
outList=[]
def outlier(df,pro,tp):
  global outList
#df - 데이터 프레임, pro - 속성, tp - 종류
  q1= df.groupby('종류')[pro].quantile(0.25)
  q3 = df.groupby('종류')[pro].quantile(0.75)
  IQR = q3 - q1
  lower = q1 - 1.5 *IQR
  upper = q3 +1.5 *IQR
  df = df[df['종류']== tp]
  df = df[ (df[pro] < lower[tp] ) | ( df[pro] > upper[tp] ) ]
  outList.append(df.index.to_numpy().tolist()) #데이터프레임의 값을 넘파이로 변경한 후 리스트로 바꾸어  outList에 저장하기

  return df

(4) 이상치 제거하기

In [ ]:
df3 = df3.drop(index = [3, 5, 6, 10, 13, 14, 18, 20, 22, 23, 24, 35, 43, 44,
                        98, 106], axis = 0) # 제거하려는 인덱스 설정

산점도로 속성 간의 관계 알아보기

In [ ]:
sns.pairplot(df3, hue = '종류', markers = ['o','s','D'])
plt.show()

히트맵을 활용하여 속성 간의 관계 알아보기

In [ ]:
irisData = df3.corr(method = 'pearson', numeric_only = True)
plt.figure(figsize = (10, 10))
sns.heatmap(data = irisData, annot = True, fmt = '.2f', linewidth = .5,
cmap = 'RdYlGn_r')
plt.show()

# 76쪽 모델 학습과 성능 평가

독립 변수와 종속변수 구분하기

In [ ]:
X = df3[['꽃잎 길이', '꽃잎 너비']] # 독립 변수(핵심 속성)
y = df3['종류'] # 종속변수

훈련 데이터와 테스트 데이터 나누기

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size = 0.2,
random_state = 7)

k-최근접 이웃 모델 생성 및 학습 진행하기

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
model = KNeighborsClassifier( ) # k-최근접 이웃 모델 생성

In [ ]:
model.fit(X_train, y_train)

테스트 데이터로 정확도 확인하기

In [ ]:
acc = model.score(X_test, y_test) # 모델의 정확도를 계산하여 저장
print('테스트 데이터 점수: {:.3f}'.format(acc)) # 정확도(소수점 3자리 출력)

혼동행렬로 성능 확인하기

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
predicted = model.predict(X_test) # 테스트 데이터에 대한 예측값
classNames = pd.unique(df3['종류'])
cm = confusion_matrix(y_test, predicted, labels=classNames) # 혼동행렬(실젯값과 예측값)
df_cm = pd.DataFrame(cm, index = [i for i in classNames],
columns = [i for i in classNames])
sns.heatmap(data = df_cm, annot = True, fmt = '0.2f', cmap = 'YlGnBu')
plt.xlabel('예측값')
plt.ylabel('실젯값')
plt.show( )

모델 개선하기

In [ ]:
import numpy as np
acc = []
k_range = range(1, 20)  # 정확도를 저장할 리스트 변수

for k in k_range: # 이웃의 수(k)를 1~ 19까지 변화시키기 위한 반복문
  model = KNeighborsClassifier(n_neighbors = k) # 이웃의 수를 k로 지정
  model.fit(X_train, y_train)
  acc.append(model.score(X_test, y_test))

plt.scatter(k_range, acc) # num과 acc의 관계 산점도
plt.xlabel('이웃의 수') # x축 라벨
plt.ylabel('정확도') # y축 라벨
plt.show( )

모델 활용하기

In [ ]:
new = [[1.0, 1.5]]
print(model.predict(new) )